# Entrenamiento YOLOv8-seg unificado (CubiCasa5K + FloorPlanCAD) en Kaggle

Antes de correr:
1. Panel derecho -> **Add Input** -> Your Datasets -> `unified-train-floorplans`.
2. Panel derecho -> Session options: **Accelerator = GPU T4 x2**, **Internet = On**.
3. Primera prueba con `EPOCHS = 1` (celda 5). Si arranca bien, poné `EPOCHS = 40` y lanzá
   **Save Version -> Save & Run All (Commit)** para que corra en segundo plano.
4. Al terminar: pestaña **Output** -> `runs/unified/weights/best.pt`.

In [ ]:
!nvidia-smi
!pip install -q ultralytics

In [ ]:
import os
DATA_ROOT = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "images" in dirs and "labels" in dirs:
        DATA_ROOT = root
        break
assert DATA_ROOT, "No encontre el dataset: agregalo en Add Input -> Your Datasets"
print("Dataset en:", DATA_ROOT)
print(os.listdir(DATA_ROOT))

In [ ]:
import yaml
names = ['wall','door','window','column','stairs','room','bathroom','kitchen','living_room',
         'bedroom','dining_room','outdoor','toilet','sink','bathtub','shower','wardrobe','cabinet',
         'kitchen_counter','fireplace','electrical_appliance','bench','misc_furniture','bed','sofa',
         'table','chair','refrigerator','gas_stove']
cfg = {"path": DATA_ROOT, "train": "images/train", "val": "images/val",
       "names": {i: n for i, n in enumerate(names)}}
yaml.safe_dump(cfg, open("/kaggle/working/data.yaml", "w"))
print(len(names), "clases")

In [ ]:
from ultralytics import YOLO

EPOCHS = 1   # prueba rapida. Para el entrenamiento real: 40

model = YOLO("yolov8s-seg.pt")
model.train(
    data="/kaggle/working/data.yaml",
    epochs=EPOCHS, imgsz=640, batch=16, patience=10, device=0, workers=4,
    project="/kaggle/working/runs", name="unified", exist_ok=True, save_period=5,
)

In [ ]:
import glob
print(glob.glob("/kaggle/working/runs/unified/weights/*.pt"))